# Session 3b: reasoning-step sweep (2 / 3 / 8 steps)

Explicit vs Implicit Memory (MPR, KDD '26) study repo. Runs on a **free-tier Colab T4** (Runtime > Change runtime type > T4 GPU).

The sweep was run on 2026-11-02 with the `colab` CLI driver (`session3b.sh`), not with this notebook; results are in `results/t4/s3_steps/`. The notebook is the same jobs for a Colab VM, committed unexecuted.

What it does: clone the repo, install vLLM, fetch the upstream code (nuster1128/MPR at commit `a24953c`, only to read its
prompt templates) and the dataset, serve `Qwen/Qwen2.5-7B-Instruct-AWQ` with vLLM, run this session's jobs, write the markdown
summaries, and save the raw results to `/content` (and to Google Drive if you set `SAVE_TO_DRIVE`). All jobs are resumable: a
finished question is skipped when you rerun a cell after the session was cut.

Folder README (experiment, provenance, expected numbers): `experiments/s3b_t4_step_sweep/README.md`.

In [ ]:
# ---- settings (no credentials here) ----
REPO_URL = (
    "https://github.com/xphiwork-dot/mpr-study.git"  # private repo: see the clone cell
)
REPO_REF = "main"
SAVE_TO_DRIVE = False  # True: mount Google Drive, keep a copy of the raw results there and resume from it
DRIVE_DIR = "/content/drive/MyDrive/mpr_study_results"
SESSION = "s3b"
LIMIT = 0  # >0: only the first N questions per run (shorter session, as LIMIT=150 in session3b.sh)

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import base64, os, shutil, subprocess

REPO = "/content/mpr-study"
if os.path.isdir(os.path.join(REPO, "experiments")):
    print("repo already at", REPO)  # e.g. uploaded by hand: nothing to clone
else:
    cmd = ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO]
    try:
        from google.colab import (
            userdata,
        )  # optional Colab secret GITHUB_TOKEN (read-only token), never stored in this notebook

        token = userdata.get("GITHUB_TOKEN")
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        cmd[1:1] = ["-c", f"http.extraheader=Authorization: Basic {basic}"]
    except Exception:
        pass  # public repo or no secret: plain clone
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-300:], (r.stderr[-300:] if r.returncode else ""))
    assert r.returncode == 0, (
        "clone failed: set the GITHUB_TOKEN secret (private repo) or upload the repo folder to /content/mpr-study"
    )
print(sorted(os.listdir(REPO)))

In [ ]:
# vLLM brings its own torch. Do NOT import vllm or torch in this kernel: that opens a CUDA context (~3.5 GB) and
# vLLM then runs out of memory on its first request. Check for the package with find_spec only.
!pip -q install uv gdown
!uv pip install --system -q vllm bm25s==0.2.9 sentence-transformers
!uv pip uninstall --system -q torchao || true      # Colab's torchao is refused by peft; not needed here
import importlib.util

print("vllm installed:", importlib.util.find_spec("vllm") is not None)

In [ ]:
import glob, zipfile

T4 = "/content/t4"  # same layout as the colab-CLI driver (experiments/_shared/t4.sh)
for d in ("scripts/t4", "logs", "results"):
    os.makedirs(f"{T4}/{d}", exist_ok=True)
# flat scripts/t4/<name>.py on the VM, as t4.sh does
for f in (
    glob.glob(f"{REPO}/experiments/_shared/*.py")
    + glob.glob(f"{REPO}/experiments/s*/*.py")
    + [f"{REPO}/experiments/00_probes_retrieval_leak/leak_check.py"]
):
    shutil.copy(f, f"{T4}/scripts/t4/")
shutil.copytree(f"{REPO}/results/t4/slices", f"{T4}/results/slices", dirs_exist_ok=True)

# upstream code: only its prompt templates are read (baselines/prompts.py); never copied into the repo
if not os.path.isdir(f"{T4}/upstream/MPR/.git"):
    os.makedirs(f"{T4}/upstream", exist_ok=True)
    !git clone -q https://github.com/nuster1128/MPR {T4}/upstream/MPR
!git -C {T4}/upstream/MPR checkout -q a24953c

# dataset (about 9 MB zip, the public Google Drive file named in the authors' README)
if not os.path.isdir(f"{T4}/data/mpr_dataset/question_answer"):
    os.makedirs(f"{T4}/data", exist_ok=True)
    !gdown 1a0Ya9buk1I0hsGoaRX8bW0QJjsKuN4FR -O /content/t4_dataset.zip
    zipfile.ZipFile("/content/t4_dataset.zip").extractall(f"{T4}/data")
print(sorted(os.listdir(f"{T4}/data")), sorted(os.listdir(f"{T4}/results")))

In [ ]:
# Optional: Google Drive keeps the raw results between Colab sessions, so a fresh VM resumes where the last one stopped.
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    prev = f"{DRIVE_DIR}/mpr_{SESSION}_results.tgz"
    if os.path.exists(prev):
        !tar xzf {prev} -C {T4}
        print("restored", prev)

In [ ]:
import time, urllib.request

MODEL_REV = "b25037543e9394b818fdfca67ab2a00ecc7dd641"  # Qwen/Qwen2.5-7B-Instruct-AWQ, pinned as in t4.sh


def serve():
    """Start vllm serve in the background and wait until /health answers (cached weights: about 2 minutes)."""
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3)
        print("server already healthy")
        return
    except Exception:
        pass
    cmd = (
        "vllm serve Qwen/Qwen2.5-7B-Instruct-AWQ --revision "
        + MODEL_REV
        + " --served-model-name qwen --dtype float16 "
        "--quantization awq --max-model-len 8192 --gpu-memory-utilization 0.80 --max-num-seqs 64 "
        "--generation-config vllm --port 8000 --host 127.0.0.1"
    )
    subprocess.Popen(
        cmd + f" > {T4}/logs/vllm.log 2>&1", shell=True, start_new_session=True
    )
    t0 = time.time()
    while time.time() - t0 < 900:
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3)
            print(f"vLLM healthy after {time.time() - t0:.0f} s")
            return
        except Exception:
            time.sleep(15)
    raise RuntimeError(
        "vLLM did not become healthy in 15 minutes: see /content/t4/logs/vllm.log"
    )


def unserve():
    subprocess.run(
        "pkill -f 'vllm serve'; pkill -f EngineCore; sleep 5; nvidia-smi --query-gpu=memory.used --format=csv,noheader",
        shell=True,
    )


def save():
    """Pack raw results and logs into /content (and Drive when SAVE_TO_DRIVE); called after every job."""
    out = f"/content/mpr_{SESSION}_results.tgz"
    subprocess.run(
        f"cd {T4} && tar czf {out} results logs/*.log logs/*.exit 2>/dev/null",
        shell=True,
    )
    if SAVE_TO_DRIVE:
        shutil.copy(out, DRIVE_DIR)
    print("saved", out)


def job(name, script, args=""):
    """Run scripts/t4/<script> on this VM in the background (survives a closed tab), poll its log, then save results."""
    exitf = f"{T4}/logs/{name}.exit"
    if os.path.exists(exitf):
        os.remove(exitf)
    cmd = (
        f"cd {T4} && JAX_PLATFORMS=cpu XLA_PYTHON_CLIENT_PREALLOCATE=false T4_ROOT={T4} "
        f"python scripts/t4/{script} {args} > logs/{name}.log 2>&1; echo $? > logs/{name}.exit"
    )
    subprocess.Popen(cmd, shell=True, start_new_session=True)
    print("launched", name, script, args)
    while not os.path.exists(exitf):
        time.sleep(60)
        print(
            "  ", "".join(open(f"{T4}/logs/{name}.log").readlines()[-1:])[-200:].strip()
        )
    rc = int(open(exitf).read().strip())
    print(f"job {name} finished rc={rc}")
    save()
    assert rc == 0, f"job {name} failed: see {T4}/logs/{name}.log"

## 1. Retrieval pass (no LLM)

`retrieve.py` embeds every user's statements with e5-base-v2 (the cache used by the dense-memory runs) and writes per-question
recall for all 12 users. It needs the GPU **without** the vLLM server, so it runs first; a user that is already done is skipped.

In [ ]:
job("retrieve", "retrieve.py")

## 2. Start vLLM

Qwen2.5-7B-Instruct AWQ (4-bit), fp16 compute, `--max-model-len 8192`; the runner uses greedy decoding.

In [ ]:
serve()

## 3. Run this session's jobs

Each cell is one call of the same script and arguments as in the session shell driver. Rerun a cell after a cut session: finished questions are skipped.

In [ ]:
LIMIT_ARG = f"--limit {LIMIT}" if LIMIT else ""
job(
    "grid3_s2",
    "grid.py",
    "--slice results/slices/s306.json --reason seq --max-step 2 --memories sparse,dense --topk 20 --tag s3_steps --workers 48 "
    + LIMIT_ARG,
)

In [ ]:
LIMIT_ARG = f"--limit {LIMIT}" if LIMIT else ""
job(
    "grid3_s3",
    "grid.py",
    "--slice results/slices/s306.json --reason seq --max-step 3 --memories sparse,dense --topk 20 --tag s3_steps --workers 48 "
    + LIMIT_ARG,
)

In [ ]:
LIMIT_ARG = f"--limit {LIMIT}" if LIMIT else ""
job(
    "grid3_s8",
    "grid.py",
    "--slice results/slices/s306.json --reason seq --max-step 8 --memories sparse,dense --topk 20 --tag s3_steps --workers 48 "
    + LIMIT_ARG,
)

In [ ]:
unserve()  # free the GPU

## 4. Summaries

Copy the raw results into the cloned repo's `results/t4/` and run the summariser there. It reads `results/t4/<tag>/*.jsonl`; the clone
already holds the earlier sessions' committed results, which the comparisons need. Output: `results/t4/summary_*.md` in the clone.

In [ ]:
os.makedirs(f"{REPO}/results/t4", exist_ok=True)
if os.path.exists(f"{T4}/results/s3_steps"):
    shutil.copytree(
        f"{T4}/results/s3_steps", f"{REPO}/results/t4/s3_steps", dirs_exist_ok=True
    )
!cd {REPO} && python3 experiments/_shared/summarize.py grid --tag s3_steps --slice results/t4/slices/s306.json
!cd {REPO} && python3 experiments/_shared/summarize.py steps
print([f for f in sorted(os.listdir(f"{REPO}/results/t4")) if f.startswith("summary_")])

## 5. Save

The raw results and logs are in `/content/mpr_<session>_results.tgz` (and on Drive if `SAVE_TO_DRIVE`). To commit them to the repo,
download the archive (uncomment the last line) and unpack it under `results/t4/` locally. Release the GPU when done
(Runtime > Disconnect and delete runtime).

In [ ]:
save()
summary = f"{REPO}/results/t4/summary_s3_steps_sweep.md"
if os.path.exists(summary):
    shutil.copy(summary, "/content/")
# from google.colab import files; files.download(f"/content/mpr_{SESSION}_results.tgz")   # uncomment to download